# Named Entity Recognition — Portuguese Regional News Articles

**Pipeline:**
1. **Load & Prep** — Read all `articles.jsonl` files from `../data-scraping/data/`
2. **NER Model** — Use spaCy's `pt_core_news_lg` for fast CPU-based NER (~100-300 art/s)
3. **Batch Processing** — Process articles with tqdm progress bar, extracting PER, ORG, LOC, MISC entities
4. **Output** — Save results as JSONL: `{article_id, entity_name, entity_type}`

**Estimated time:** ~3–10 hours for 3.5M articles on CPU

## Step 0: Install Dependencies

In [14]:
# %pip install polars spacy tqdm
# !python -m spacy download pt_core_news_lg

## Step 1: Load Articles

In [15]:
import polars as pl
from pathlib import Path
import time

DATA_DIR = Path("../data-scraping/data")
JORNAIS_CSV = Path("../data-scraping/jornais.csv")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Load the list of known newspapers and extract their domains
jornais = pl.read_csv(JORNAIS_CSV)
known_domains = set(
    jornais["url"]
    .drop_nulls()
    .str.replace(r"^https?://", "")
    .str.replace(r"^www\.", "")
    .str.strip_chars("/")
    .to_list()
)
print(f"Known domains from jornais.csv: {len(known_domains)}")

# Scan all articles.jsonl files
t0 = time.time()
jsonl_files = sorted(DATA_DIR.glob("*/articles.jsonl"))
print(f"Found {len(jsonl_files)} articles.jsonl files")

ALL_UTF8 = {col: pl.Utf8 for col in [
    "id", "url", "title", "subtitle", "text", "date", "author",
    "agency", "section", "source", "timestamp", "archive_url",
    "domain", "text_hash", "fetched_at", "extractor",
]}

dfs = []
skipped = 0
for f in jsonl_files:
    try:
        df = pl.read_ndjson(f, schema_overrides=ALL_UTF8).select(
            ["id", "title", "text", "domain"]
        )
        dfs.append(df)
    except Exception as e:
        skipped += 1

articles = pl.concat(dfs)
elapsed = time.time() - t0
print(f"Loaded {len(articles):,} articles in {elapsed:.1f}s (skipped {skipped} files)")

Known domains from jornais.csv: 713
Found 537 articles.jsonl files
Loaded 3,517,501 articles in 15.5s (skipped 0 files)


In [16]:
import gc
del dfs, jsonl_files, jornais
gc.collect()
print(f"Ready for NER. Articles shape: {articles.shape}")

Ready for NER. Articles shape: (3517501, 4)


In [17]:
# Filter to only known domains
articles = articles.filter(
    pl.col("domain")
    .str.replace(r"^www\.", "")
    .str.strip_chars("/")
    .is_in(known_domains)
)
print(f"Articles from known jornais: {len(articles):,}")

# Combine title + text, drop near-empty articles
articles = articles.with_columns(
    (pl.col("title").fill_null("") + " " + pl.col("text").fill_null(""))
    .str.strip_chars()
    .alias("full_text")
).filter(pl.col("full_text").str.len_chars() > 50)

# Keep only id and full_text for NER
articles = articles.select(["id", "full_text"])
print(f"Articles after filtering: {len(articles):,}")
articles.head(3)

Articles from known jornais: 3,476,505
Articles after filtering: 3,476,505


id,full_text
str,str
"""88caa7c06701d4aababb797ffede50…","""21TV e SAPO.PT: Juntas pela In…"
"""9136a340d4cf545335eaf43db36e11…","""Incidência semanal de fiscaliz…"
"""9dfda7664d7ee9eb4cf8326fdddabb…","""Violência doméstica com recurs…"


In [18]:
del known_domains
gc.collect()
print(f"Ready for NER. Articles shape: {articles.shape}")

Ready for NER. Articles shape: (3476505, 2)


## Step 2: Load NER Model

Using spaCy's `pt_core_news_lg` — a Portuguese language model trained on news data. Much faster than transformer models on CPU (~100-300 articles/sec).

Entity types:
- **PER** — Person names
- **ORG** — Organizations
- **LOC** — Locations
- **MISC** — Miscellaneous entities

In [19]:
import spacy

# Load Portuguese NER model (disable components we don't need for speed)
nlp = spacy.load("pt_core_news_lg", disable=["tagger", "parser", "lemmatizer", "attribute_ruler"])

# Increase max length for long articles
nlp.max_length = 3_000_000

print(f"Loaded model: pt_core_news_lg")
print(f"Pipeline components: {nlp.pipe_names}")
print(f"NER labels: {nlp.get_pipe('ner').labels}")

Loaded model: pt_core_news_lg
Pipeline components: ['tok2vec', 'morphologizer', 'ner']
NER labels: ('LOC', 'MISC', 'ORG', 'PER')


## Step 3: Quick Sanity Check

In [20]:
# Test on a sample article
sample_text = articles[0, "full_text"][:2000]
doc = nlp(sample_text)

print(f"Sample text (first 200 chars): {sample_text[:200]}...")
print(f"\nEntities found: {len(doc.ents)}")
for ent in doc.ents[:15]:
    print(f"  {ent.label_:5s} | {ent.text}")

Sample text (first 200 chars): 21TV e SAPO.PT: Juntas pela Informação e Entretenimento na Região Oeste e Grande Lisboa A 21TV, uma WEBTV da região Oeste e Área Metropolitana de Lisboa, e o SAPO.PT, plataforma conceituada de notícia...

Entities found: 14
  MISC  | SAPO.PT: Juntas pela Informação e Entretenimento
  LOC   | Região Oeste
  LOC   | Grande Lisboa
  LOC   | Oeste
  LOC   | Área Metropolitana de Lisboa
  ORG   | SAPO.PT
  LOC   | Região Oeste
  LOC   | Grande Lisboa
  ORG   | SAPO.PT
  ORG   | SAPO.PT
  LOC   | região Oeste
  LOC   | Grande Lisboa
  ORG   | SAPO.PT
  ORG   | SAPO.PT


## Step 4: Batch NER Processing

Process all articles using spaCy's `nlp.pipe()` for efficient batching. For each article we:
1. Truncate text to first 2000 chars (NER is most useful on the lead paragraphs)
2. Run NER via spaCy pipe (multi-threaded)
3. Deduplicate entities per article
4. Write results incrementally to JSONL

Progress tracked with tqdm — expect ~100-300 articles/sec on CPU.

In [21]:
import json
from tqdm.auto import tqdm

OUTPUT_FILE = OUTPUT_DIR / "entities.jsonl"
MAX_CHARS = 2000
BATCH_SIZE = 1000  # spaCy pipe batch size
N_PROCESS = 4     # parallel workers for spaCy

# Check for existing progress (resume support)
processed_ids: set = set()
if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE) as f:
        for line in f:
            row = json.loads(line)
            processed_ids.add(row["article_id"])
    print(f"Resuming: {len(processed_ids):,} articles already processed")

# Filter out already-processed articles
remaining = articles.filter(~pl.col("id").is_in(list(processed_ids)))
print(f"Articles to process: {len(remaining):,}")

Resuming: 5,790 articles already processed
Articles to process: 3,470,715


In [22]:
t0 = time.time()
total = len(remaining)
n_entities_total = 0

CHUNK_SIZE = 2000  # rows to materialize at a time

with open(OUTPUT_FILE, "a") as out_f:
    for chunk_start in tqdm(range(0, total, CHUNK_SIZE), desc="NER chunks", unit="chunk"):
        chunk = remaining.slice(chunk_start, CHUNK_SIZE)
        chunk_ids = chunk["id"].to_list()
        chunk_texts = [t[:MAX_CHARS] for t in chunk["full_text"].to_list()]

        for i, doc in enumerate(nlp.pipe(chunk_texts, batch_size=256)):
            article_id = chunk_ids[i]

            # Deduplicate entities per article (same name + type)
            seen = set()
            for ent in doc.ents:
                name = ent.text.strip()
                etype = ent.label_

                if len(name) < 2:
                    continue

                key = (name.lower(), etype)
                if key in seen:
                    continue
                seen.add(key)

                out_f.write(json.dumps({
                    "article_id": article_id,
                    "entity_name": name,
                    "entity_type": etype,
                }, ensure_ascii=False) + "\n")
                n_entities_total += 1

        # Free chunk memory
        del chunk, chunk_ids, chunk_texts
        gc.collect()

elapsed = time.time() - t0
print(f"\nDone! Processed {total:,} articles in {elapsed/60:.1f} minutes")
print(f"Rate: {total/elapsed:.0f} articles/sec")
print(f"Total entities extracted: {n_entities_total:,}")
print(f"Output: {OUTPUT_FILE}")

NER chunks:   0%|          | 0/1736 [00:00<?, ?chunk/s]


Done! Processed 3,470,715 articles in 938.4 minutes
Rate: 62 articles/sec
Total entities extracted: 47,927,839
Output: output/entities.jsonl


## Step 5: Quick Analysis

In [ ]:
# Load and analyze results
entities_df = pl.read_ndjson(OUTPUT_FILE)
print(f"Total entity records: {len(entities_df):,}")
print(f"Unique articles: {entities_df['article_id'].n_unique():,}")
print(f"Unique entities: {entities_df['entity_name'].n_unique():,}")

print("\nEntity type distribution:")
type_counts = entities_df.group_by("entity_type").len().sort("len", descending=True)
for row in type_counts.iter_rows():
    print(f"  {row[0]:5s}: {row[1]:>10,}")

print("\nTop 20 most mentioned entities:")
top_entities = (
    entities_df.group_by(["entity_name", "entity_type"])
    .len()
    .sort("len", descending=True)
    .head(20)
)
for row in top_entities.iter_rows():
    print(f"  {row[1]:5s} | {row[2]:>6,} | {row[0]}")

Total entity records: 48,007,525
Unique articles: 3,415,367
Unique entities: 6,082,711

Entity type distribution:
  LOC  : 20,566,477
  MISC :  9,972,831
  ORG  :  9,080,686
  PER  :  8,387,531

Top 20 most mentioned entities:


In [ ]:
# Sample output rows
print("Sample output (first 10 rows):")
print(entities_df.head(10))